# Clase 024 — Operaciones y alineación

**Parte 0** · VanderPlas cap. 3 § 3.4.

> 🎯 Alineación por index, apply vs vectorización, fill_value.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import time

## 1️⃣ Alineación automática

Operar dos Series alinea por **index**, no por posición:

In [ ]:
a = pd.Series([100, 200, 300], index=['x', 'y', 'z'])
b = pd.Series([10, 20, 30],    index=['y', 'z', 'w'])

print('a:'); print(a)
print('\nb:'); print(b)
print('\na + b — alinea, NaN donde no hay match:')
print(a + b)

## 2️⃣ `fill_value` evita propagar NaN

In [ ]:
print('a.add(b, fill_value=0):')
print(a.add(b, fill_value=0))
# x: 100+0=100  y: 200+10=210  z: 300+20=320  w: 0+30=30

## 3️⃣ `apply` — flexible pero lento por fila

**Regla**: si puedes hacerlo con ufunc/operadores vectorizados, **no uses apply**. Si necesitas lógica compleja por fila, sí.

In [ ]:
df = pd.DataFrame({
    'masa': [3750, 3800, 3250, 4400, 3700],
    'pico': [39.1, 39.5, 40.3, 36.7, 39.3],
})

# apply axis=1: una fila por iteración (lento)
def bmi_fila(row):
    return row['masa'] / (row['pico'] ** 2)

bmi_apply = df.apply(bmi_fila, axis=1)
print('con apply:')
print(bmi_apply.round(3))

# Vectorizado: una operación sobre todo el array (rápido)
bmi_vec = df['masa'] / (df['pico'] ** 2)
print('\nvectorizado:')
print(bmi_vec.round(3))
print(f'\niguales? {(bmi_apply.round(6) == bmi_vec.round(6)).all()}')

## 4️⃣ Benchmark apply vs vectorizado

In [ ]:
rng = np.random.default_rng(42)
grande = pd.DataFrame({
    'masa': rng.uniform(3000, 5000, 10_000),
    'pico': rng.uniform(35, 50, 10_000),
})

t0 = time.perf_counter(); grande.apply(bmi_fila, axis=1); t1 = time.perf_counter()
t2 = time.perf_counter(); grande['masa'] / (grande['pico'] ** 2); t3 = time.perf_counter()

print(f'apply        : {(t1-t0)*1000:.1f} ms')
print(f'vectorizado  : {(t3-t2)*1000:.2f} ms')
print(f'speedup      : {(t1-t0)/(t3-t2):.0f}×')

## 5️⃣ `map` para Series — recodificación con dict

Útil para mapear categorías a códigos o relabelar:

In [ ]:
species = pd.Series(['Adelie', 'Chinstrap', 'Gentoo', 'Adelie', 'Gentoo'])
codigo = species.map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})
print(pd.DataFrame({'species': species, 'codigo': codigo}))

## 6️⃣ `df.map` — elementwise (era `applymap`)

Aplica una función a **cada celda** del DataFrame. Lento — úsalo solo cuando vectorización no aplica:

In [ ]:
df_num = pd.DataFrame({'A': [1.234, 5.678], 'B': [9.0, 0.1234]})
format_pct = df_num.map(lambda x: f'{x*100:.2f}%')
print(format_pct)

## 7️⃣ ufuncs NumPy preservan index

Pandas "sabe" NumPy — aplicar `np.log`, `np.sqrt`, etc., a una Series mantiene el index:

In [ ]:
s = pd.Series([1, 10, 100, 1000], index=['a', 'b', 'c', 'd'])
print('s:'); print(s)
print('\nnp.log(s):'); print(np.log(s).round(3))

## ✅ Checklist

- [ ] Sé que pandas alinea por index automáticamente
- [ ] Uso `fill_value` para evitar NaN en operaciones
- [ ] Prefiero vectorización a apply
- [ ] Uso `map` para recodificar Series con dict
- [ ] Sé que ufuncs NumPy preservan el index

## 📝 Homework

Ver `README.md`. BMI con apply vs vectorizado + benchmark, map species, alineación con fill_value.

## 📖 Definiciones y características

**Alineación por index**

Operación matemática entre dos pandas (Series + Series, DF + Series, etc.) alinea por **etiqueta** de index. Labels que no estén en ambos → NaN.

**`fill_value` en operaciones**

Parámetro que reemplaza NaN durante la operación: `s1.add(s2, fill_value=0)` trata índices ausentes como 0 en vez de propagar NaN.

**`apply`**

Aplica función a cada fila (`axis=1`) o columna (`axis=0`) de un DataFrame, o cada elemento de una Series. **Lento** porque itera en Python. Usa solo si vectorización no aplica.

**`map` (Series)**

Aplica función o **dict** a cada elemento. Útil para recodificación rápida: `s.map({'A': 1, 'B': 2})`. Más rápido que `apply` por su API más restringida.

**`df.map` (antes `applymap`)**

Aplica función a **cada celda** del DataFrame (elementwise). Lento; usa solo cuando vectorización no aplica. `applymap` está deprecated en pandas 2.1+.

**Ufunc-aware**

Pandas Series respeta ufuncs NumPy: `np.log(s)`, `np.sqrt(s)` funcionan y **preservan el index**. Ventaja sobre convertir a array y perder labels.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `apply` toma 10× más que vectorización equivalente | Iteración Python por fila. **Fix**: piensa si tu lógica es expresable con operaciones vectorizadas (`df['a'] * df['b']`); 99% de los `apply` simples se pueden eliminar. |
| `s1 + s2` produce NaN aunque los datos están completos | Index distinto (incluso por orden diferente). **Fix**: alinea antes (`s2 = s2.reindex(s1.index)`) o usa `s1.add(s2, fill_value=0)`. |
| `df.apply(func, axis=1)` rompe con `KeyError` | Tu función accede `row['col_x']` pero esa col no existe. **Fix**: imprime `row.index` dentro de la función para ver qué hay. |
| `df.map(lambda x: ...)` falla "object has no attribute 'map'" | Es método de DataFrame solo en pandas ≥2.1. Para versiones viejas: `df.applymap(...)`. Para Series: `s.map(...)`. |
| `np.log(df['x'])` da error con NaN | Si NaN está presente, log da NaN. Si negativos, lanza warning y NaN. **Fix**: filtra primero (`df[df['x'] > 0]`) o usa `np.log1p` para log(1+x). |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo `apply` y cuándo NO?**

**NO**: si lo puedes hacer con `df['a'] * df['b']` u operación pandas built-in (groupby, transform, etc.). **SÍ**: lógica compleja por fila que no se descompone.

**❓ ¿`apply(axis=1)` con tipos mixtos da problemas?**

Sí — pandas convierte cada row a Series con dtype común. Si tienes `int` y `str`, queda `object` y los operadores `<`, `>` rompen. Mejor extrae columnas y opera directo.

**❓ ¿Cómo paralelizo apply?**

`pandarallel`, `swifter`, `modin` — drop-in replacements. Pero antes de paralelizar, asegúrate que tu apply no es vectorizable (el speedup es mayor).

**❓ ¿`s.map(dict)` o `s.replace(dict)`?**

**`map`**: mapea uno-a-uno; valores no encontrados en dict → NaN. **`replace`**: reemplaza solo los que aparecen; el resto queda igual. Elige según comportamiento deseado en valores faltantes.

**❓ ¿Por qué pandas a veces es más lento que NumPy?**

Overhead del index, manejo de NaN, dtype-aware. Para operaciones puramente numéricas sobre datos limpios y rectangulares, NumPy puede ser 2-5× más rápido. Pandas vale por la API, no por velocidad raw.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.4
- [pandas function application](https://pandas.pydata.org/docs/user_guide/basics.html#function-application)

➡️ **Siguiente:** [025 — Datos faltantes](../025-pandas-datos-faltantes/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Suma con alineación** (default y `fill_value=0`).

In [ ]:
import pandas as pd, numpy as np
s1 = pd.Series({'a': 1, 'b': 2, 'c': 3})
s2 = pd.Series({'b': 10, 'c': 20, 'd': 30})
print('default (NaN):\n', s1 + s2)
print('fill_value=0:\n', s1.add(s2, fill_value=0))
assert s1.add(s2, fill_value=0)['a'] == 1

**Ej. 2 — `apply` por fila** (BMI = masa / bill_length²).

In [ ]:
import numpy as np, pandas as pd

def make_penguins(seed=42, with_na=False):
    """DataFrame sintetico estilo Palmer Penguins (344 filas), sin internet."""
    rng = np.random.default_rng(seed)
    cfg = {  # especie: (n, islas, bill_len, bill_depth, flipper, body_mass)
        'Adelie':    (152, ['Torgersen', 'Biscoe', 'Dream'], 38.8, 18.3, 190, 3700),
        'Chinstrap': (68,  ['Dream'],                        48.8, 18.4, 196, 3733),
        'Gentoo':    (124, ['Biscoe'],                       47.5, 15.0, 217, 5076),
    }
    filas = []
    for sp, (n, islas, bl, bd, fl, bm) in cfg.items():
        for _ in range(n):
            sex = rng.choice(['male', 'female'])
            k = 1.0 if sex == 'male' else 0.93
            filas.append({
                'species': sp,
                'island': rng.choice(islas),
                'bill_length_mm': round(float(rng.normal(bl, 2.5)), 1),
                'bill_depth_mm': round(float(rng.normal(bd, 1.2)), 1),
                'flipper_length_mm': float(round(rng.normal(fl, 6))),
                'body_mass_g': float(round(rng.normal(bm * k, 300))),
                'sex': sex,
            })
    df = pd.DataFrame(filas)
    if with_na:
        idx = rng.choice(df.index, size=12, replace=False)
        df.loc[idx[:6], 'bill_length_mm'] = np.nan
        df.loc[idx[6:], 'sex'] = np.nan
    return df

df = make_penguins()
def bmi(row):
    return row['body_mass_g'] / (row['bill_length_mm'] ** 2)
df['bmi_apply'] = df.apply(bmi, axis=1)
print(df[['body_mass_g', 'bill_length_mm', 'bmi_apply']].head())
assert df['bmi_apply'].notna().all()

**Ej. 3 — Mismo cálculo vectorizado** (y comparación de tiempos).

In [ ]:
import timeit
df['bmi_vec'] = df['body_mass_g'] / (df['bill_length_mm'] ** 2)
assert np.allclose(df['bmi_vec'], df['bmi_apply'])
t_apply = timeit.timeit(lambda: df.apply(bmi, axis=1), number=5)
t_vec   = timeit.timeit(lambda: df['body_mass_g'] / (df['bill_length_mm'] ** 2), number=5)
print(f'apply: {t_apply*1000:.1f} ms | vectorizado: {t_vec*1000:.2f} ms (mucho mas rapido)')

**Ej. 4 — `map` con dict** (species -> código).

In [ ]:
codigos = df['species'].map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})
print(codigos.value_counts())
assert set(codigos.unique()) <= {0, 1, 2}

**Ej. 5 — ufunc de NumPy preserva el index.**

In [ ]:
logm = np.log(df['body_mass_g'])
assert (logm.index == df.index).all()
print('np.log preserva el index. Primeros valores:\n', logm.head())